# 02 — Data Cleaning Indonesia News

Notebook ini membersihkan dataset **Indonesia News** dengan Polars Lazy API, menyimpan hasilnya sebagai Parquet, lalu melakukan profiling menggunakan DuckDB `SUMMARIZE`.

## Aturan kualitas data

1. Nilai kosong dan spasi berlebih pada kolom teks dinormalisasi.
2. Artikel tanpa judul, isi, atau tanggal valid dihapus.
3. Artikel dengan isi kurang dari 100 karakter dihapus karena dianggap bukan artikel lengkap.
4. Tanggal dibatasi pada 2014–2020 sesuai periode dataset.
5. Duplikasi berdasarkan `source`, `title`, dan `published_at` dihapus.
6. Tag kosong diisi dengan `Tidak diketahui`.

Data mentah pada `data/raw/` tidak diubah. Hasil cleaning ditulis ke `data/processed/indonesia_news_clean.parquet`.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(100)

print(f"Polars: {pl.__version__}")
print(f"DuckDB: {duckdb.__version__}")

Polars: 1.44.2
DuckDB: 1.5.5


## 1. Lokasi data

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "archive"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "indonesia_news_clean.parquet"

FILE_CONFIG = [
    {"name": "cnn.csv", "separator": "|", "source": "CNN Indonesia"},
    {"name": "tempo.csv", "separator": "|", "source": "Tempo"},
    {"name": "tempo010619310520.csv", "separator": ";", "source": "Tempo"},
]

missing_files = [item["name"] for item in FILE_CONFIG if not (RAW_DIR / item["name"]).exists()]
if missing_files:
    raise FileNotFoundError(f"File belum ditemukan di {RAW_DIR}: {', '.join(missing_files)}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data mentah : {RAW_DIR}")
print(f"Hasil bersih: {OUTPUT_PATH}")

Data mentah : /home/jovyan/work/data/raw/archive
Hasil bersih: /home/jovyan/work/data/processed/indonesia_news_clean.parquet


## 2. Membaca data secara lazy

Setiap file dipindai tanpa memuat seluruh dataset ke memori. Nilai waktu Unix milidetik juga dinormalisasi, termasuk nilai yang memakai koma atau notasi ilmiah.

In [3]:
EXPECTED_COLUMNS = ["content", "datetime", "tags", "title"]

def scan_news_file(config: dict) -> pl.LazyFrame:
    datetime_ms = (
        pl.col("datetime")
        .cast(pl.String)
        .str.replace_all(",", ".")
        .cast(pl.Float64, strict=False)
        .round(0)
        .cast(pl.Int64, strict=False)
    )

    return (
        pl.scan_csv(
            RAW_DIR / config["name"],
            separator=config["separator"],
            has_header=True,
            quote_char='"',
            encoding="utf8-lossy",
            infer_schema_length=10_000,
            schema_overrides={column: pl.String for column in EXPECTED_COLUMNS},
            null_values=["", "null", "None", "NA", "N/A"],
            ignore_errors=True,
            truncate_ragged_lines=True,
            low_memory=True,
        )
        .select(EXPECTED_COLUMNS)
        .with_columns(
            pl.lit(config["source"]).alias("source"),
            pl.lit(config["name"]).alias("source_file"),
            datetime_ms.alias("datetime_ms"),
        )
        .with_columns(
            pl.from_epoch(pl.col("datetime_ms"), time_unit="ms").alias("published_at")
        )
    )

raw_news = pl.concat(
    [scan_news_file(item) for item in FILE_CONFIG],
    how="vertical_relaxed",
)
raw_news.collect_schema()

Schema([('content', String),
        ('datetime', String),
        ('tags', String),
        ('title', String),
        ('source', String),
        ('source_file', String),
        ('datetime_ms', Int64),
        ('published_at', Datetime(time_unit='us', time_zone=None))])

## 3. Ringkasan sebelum cleaning

In [4]:
before_summary = raw_news.select(
    pl.len().alias("rows_before"),
    pl.col("title").is_null().sum().alias("null_title"),
    pl.col("content").is_null().sum().alias("null_content"),
    pl.col("tags").is_null().sum().alias("null_tags"),
    pl.col("published_at").is_null().sum().alias("invalid_datetime"),
).collect(engine="streaming")

before_summary

rows_before,null_title,null_content,null_tags,invalid_datetime
u32,u32,u32,u32,u32
988814,0,12870,22978,4


## 4. Pipeline cleaning dengan Polars

`strip_chars` menghapus spasi di awal/akhir, sedangkan ekspresi reguler merapikan rentetan whitespace di dalam teks. Kolom panjang teks ditambahkan untuk validasi dan analisis berikutnya.

In [5]:
def normalize_text(column: str) -> pl.Expr:
    return (
        pl.col(column)
        .cast(pl.String)
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )

MIN_DATE = pl.datetime(2014, 1, 1)
MAX_DATE = pl.datetime(2020, 12, 31, 23, 59, 59)

clean_news = (
    raw_news
    .with_columns(
        normalize_text("title").alias("title"),
        normalize_text("content").alias("content"),
        normalize_text("tags").alias("tags"),
    )
    .with_columns(
        pl.when(pl.col("tags").is_null() | pl.col("tags").eq(""))
        .then(pl.lit("Tidak diketahui"))
        .otherwise(pl.col("tags"))
        .alias("tags"),
        pl.col("title").str.len_chars().alias("title_length"),
        pl.col("content").str.len_chars().alias("content_length"),
    )
    .filter(
        pl.col("title").is_not_null()
        & pl.col("title").ne("")
        & pl.col("content").is_not_null()
        & (pl.col("content_length") >= 100)
        & pl.col("published_at").is_not_null()
        & pl.col("published_at").is_between(MIN_DATE, MAX_DATE, closed="both")
    )
    .unique(subset=["source", "title", "published_at"], keep="first", maintain_order=False)
    .select(
        "source",
        "source_file",
        "published_at",
        "title",
        "tags",
        "content",
        "title_length",
        "content_length",
    )
)

clean_news.collect_schema()

Schema([('source', String),
        ('source_file', String),
        ('published_at', Datetime(time_unit='us', time_zone=None)),
        ('title', String),
        ('tags', String),
        ('content', String),
        ('title_length', UInt32),
        ('content_length', UInt32)])

## 5. Menyimpan hasil sebagai Parquet

Parquet dipilih karena terkompresi, menyimpan tipe data, dan dapat dibaca langsung oleh Polars maupun DuckDB. Proses ini dapat membutuhkan beberapa menit karena ukuran data mentah sekitar 2 GB.

In [6]:
clean_news.sink_parquet(
    OUTPUT_PATH,
    compression="zstd",
    compression_level=3,
    statistics=True,
)

output_size_mib = OUTPUT_PATH.stat().st_size / 1024**2
print(f"Parquet tersimpan: {OUTPUT_PATH}")
print(f"Ukuran hasil   : {output_size_mib:,.2f} MiB")

Parquet tersimpan: /home/jovyan/work/data/processed/indonesia_news_clean.parquet
Ukuran hasil   : 702.34 MiB


## 6. Audit hasil cleaning

In [7]:
clean_scan = pl.scan_parquet(OUTPUT_PATH)

after_summary = clean_scan.select(
    pl.len().alias("rows_after"),
    pl.col("title").null_count().alias("null_title"),
    pl.col("content").null_count().alias("null_content"),
    pl.col("published_at").null_count().alias("null_published_at"),
    (pl.col("content_length") < 100).sum().alias("short_content"),
    pl.struct(["source", "title", "published_at"]).is_duplicated().sum().alias("duplicate_rows"),
    pl.col("published_at").min().alias("earliest_datetime"),
    pl.col("published_at").max().alias("latest_datetime"),
).collect(engine="streaming")

rows_before = before_summary.item(0, "rows_before")
rows_after = after_summary.item(0, "rows_after")
audit = pl.DataFrame({
    "metric": ["Baris sebelum", "Baris sesudah", "Baris dihapus", "Persentase dipertahankan"],
    "value": [
        f"{rows_before:,}",
        f"{rows_after:,}",
        f"{rows_before - rows_after:,}",
        f"{rows_after / rows_before * 100:.2f}%",
    ],
})

display(audit)
after_summary

metric,value
str,str
"""Baris sebelum""","""988,814"""
"""Baris sesudah""","""964,553"""
"""Baris dihapus""","""24,261"""
"""Persentase dipertahankan""","""97.55%"""


rows_after,null_title,null_content,null_published_at,short_content,duplicate_rows,earliest_datetime,latest_datetime
u32,u32,u32,u32,u32,u32,datetime[μs],datetime[μs]
964553,0,0,0,0,0,2014-08-04 08:35:00,2020-05-31 16:52:00


In [8]:
source_summary = (
    clean_scan
    .group_by("source")
    .agg(
        pl.len().alias("article_count"),
        pl.col("published_at").min().alias("earliest_datetime"),
        pl.col("published_at").max().alias("latest_datetime"),
        pl.col("content_length").median().alias("median_content_length"),
    )
    .sort("article_count", descending=True)
    .collect(engine="streaming")
)

source_summary

source,article_count,earliest_datetime,latest_datetime,median_content_length
str,u32,datetime[μs],datetime[μs],f64
"""Tempo""",632086,2014-12-31 19:54:00,2020-05-31 16:52:00,1781.0
"""CNN Indonesia""",332467,2014-08-04 08:35:00,2020-05-31 16:50:00,2044.0


## 7. Profiling DuckDB dengan `SUMMARIZE`

DuckDB membaca file Parquet secara langsung. `SUMMARIZE` menghasilkan tipe data, jumlah baris, jumlah nilai null, nilai minimum–maksimum, nilai unik perkiraan, rata-rata, kuartil, dan simpangan baku untuk setiap kolom.

In [9]:
parquet_sql_path = OUTPUT_PATH.as_posix().replace("'", "''")
con = duckdb.connect()

duckdb_summary = con.sql(
    f"SUMMARIZE SELECT * FROM read_parquet('{parquet_sql_path}')"
).pl()

duckdb_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""source""","""VARCHAR""","""CNN Indonesia""","""Tempo""",2,null,null,null,null,null,964553,0.00
"""source_file""","""VARCHAR""","""cnn.csv""","""tempo010619310520.csv""",3,null,null,null,null,null,964553,0.00
"""published_at""","""TIMESTAMP""","""2014-08-04 08:35:00""","""2020-05-31 16:52:00""",798908,"""2018-02-11 21:43:21.873593""",null,"""2016-09-13 09:55:43.295588""","""2018-05-08 23:00:19.24367""","""2019-08-21 21:53:21.097182""",964553,0.00
"""title""","""VARCHAR""","""""Amuk Munir"" Desak Pemerintah Buka Dokumen TPF Kasus Munir""","""tuhan""",757029,null,null,null,null,null,964553,0.00
"""tags""","""VARCHAR""","""#2019GantiPresiden""","""zurich,asuransi,digital""",630589,null,null,null,null,null,964553,0.00
"""content""","""VARCHAR""",""""" itu,"" kata Nicke Widyawati usai mengecek realisasi penggunaan bahan bakar minyak dengan campuran b…","""”Wanhong ini produksinya akan diekspor ke Cina. Karena memang perusahaan asal Cina. Mereka pilih Pal…",890277,null,null,null,null,null,964553,0.00
"""title_length""","""UINTEGER""","""1""","""126""",97,"""54.79839262331878""","""8.06283409663966""","""50""","""57""","""61""",964553,0.00
"""content_length""","""UINTEGER""","""100""","""52721""",7461,"""2044.9093123965195""","""912.5953484222294""","""1511""","""1872""","""2365""",964553,0.00


In [10]:
duckdb_validation = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(*) - COUNT(title) AS null_title,
        COUNT(*) - COUNT(content) AS null_content,
        COUNT(*) - COUNT(published_at) AS null_published_at,
        COUNT(*) FILTER (WHERE content_length < 100) AS short_content,
        COUNT(*) - COUNT(DISTINCT (source, title, published_at)) AS duplicate_rows
    FROM read_parquet('{parquet_sql_path}')
    """
).pl()

duckdb_validation

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

row_count,null_title,null_content,null_published_at,short_content,duplicate_rows
i64,i64,i64,i64,i64,i64
964553,0,0,0,0,0


## Kesimpulan

Dataset hasil cleaning siap dipakai oleh `03_eda_and_insights.ipynb`. Sebelum commit, pastikan seluruh sel berhasil dijalankan, hasil validasi bernilai nol untuk null wajib, artikel pendek, dan duplikasi, lalu simpan notebook dengan **Ctrl+S**. File Parquet berukuran besar tetap tidak perlu dimasukkan ke Git.